In [1]:
import os
from getpass import getpass
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

c:\Users\Playdata\Desktop\mle-01-p1-team2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").is_dir() and (PROJECT_DIR.parent / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

load_dotenv(PROJECT_DIR / ".env")
CHROMA_DIR = PROJECT_DIR.parent / "data" / "chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    encode_kwargs={"normalize_embeddings": True},
)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"ChromaDB 파일: {CHROMA_DIR / 'chroma.sqlite3'}")
print("OpenAI 답변 생성 모델 준비 완료")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2762.40it/s]


프로젝트 경로: c:\Users\Playdata\Desktop\mle-01-p1-team2\notebooks
ChromaDB 파일: c:\Users\Playdata\Desktop\mle-01-p1-team2\data\chroma_db\chroma.sqlite3
OpenAI 답변 생성 모델 준비 완료


In [3]:
CHROMA_DIR

WindowsPath('c:/Users/Playdata/Desktop/mle-01-p1-team2/data/chroma_db')

In [4]:
import chromadb
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(name="pet_care")
data = collection.get(
    limit=10,
    include=["documents", "metadatas"]
)

print(data)

{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], 'embeddings': None, 'documents': ['저희 집에서 기르고 있는 것으로 강아지는 제 팔뚝 정도 되는 작은 크기의 반려견입니다. 그런데 정확한 시기는 기억나지 않지만, 약 한 달 전 즈음에 같은 아파트에 거주하는 수컷 강아지와 교배를 하였습니다. 교배 이후 약 3주가 지나자 우리 집 강아지가 살이 찌는 모습을 보였습니다. 저는 걱정스러운 부분이 있습니다. 일반적으로 암컷이 수컷보다 약간 더 큰 것이 이상적이 라고 알고 있는 것으로 데, 저희 강아지는 그와 는 정반대의 경우로, 수컷 강아지가 2. 5배에서 3배 정도 더 큰 상황입니다. 이러한 경우, 나중에 분만할 때 어려움이 없을 지 걱정됩니다. 우리 강아지의 크기가 작아서 1마리 정도 임신할 것 같다는 생각이 드는 데, 만약 1마리를 가지게 된다면 분만 과정이 더욱 어려워지지 않을 까 염려됩니다. 언제나 힘이 없고 건강한 편이 아니라 허약한 것 같은 느낌이 드는 데, 혹시 제왕절개를 해야 하는 상황이 라면 우리 강아지가 잘 버텨줄 수 있을 지에 대해 자세히 답변해 주시면 감사하겠습니다.', '저희 집에서 기르고 있는 진돗개가 최근 설사를 하는 증상을 보이고 있습니다. 특히 우려스러운 점은 강아지가 항문을 자주 핥는 행동을 하고 있다는 것입니다. 현재 거주하고 있는 지역이 시골이라 동물병원까지의 거리가 상당히 멀기 때문에 즉시 병원에 방문하기가 어려운 상황입니다. 이런 경우, 반드시 병원에 가야 하는지, 아니면 자택에서 약물로 처리할 수 있는 방법이 있는지 궁금합니다. 참고로, 강아지의 변 상태는 약간 녹색빛이 감도는 색이며, 물처럼 묽지는 않지만 점도가 있는 형태입니다. 이러한 증상에 대해 어떻게 적절히 대처해야 할지 조언 부탁드립니다.', '항문낭을 관리하지 않다가 몇일 전, 제가 앉아 있는 동안 제 하체에 강아지가 앉아 있었고, 그때 갑작스레 이상한 냄새가 나는 것을 느꼈습니다.

In [5]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """아래 [검색 데이터]를 근거로 사용자의 질문에 답하세요.
규칙:
1. 검색된 데이터에 근거해서만 답변하세요.
2. 데이터에 없는 내용은 임의로 추측하지 마세요.
3. 답변은 간결하게 작성하세요.

[검색 데이터]
{context}
""",
    ),
    ("human", "{question}"),
])
rag_chain = prompt | model | parser if model is not None else None

In [6]:
vector_db = Chroma(
    collection_name="pet_care",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

In [7]:
def ask_rag(question, k=3):
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join(
        f"질문: {doc.page_content}\n답변: {doc.metadata.get('qa.output', '')}"
        for doc in docs
    )

    if rag_chain is None:
        answer = "유사도 검색은 성공했습니다. 답변 생성에는 OPENAI_API_KEY가 필요합니다."
    else:
        answer = rag_chain.invoke({"context": context, "question": question})

    evidence_rows = [doc.metadata for doc in docs]
    return {"answer": answer, "evidence_rows": evidence_rows}

In [8]:
result = ask_rag("우리 말티즈가 항문을 계속 핥는데 뭐가 문제야?") 
print("답변:") 
print(result["answer"]) 

print("\n근거 데이터 row:") 
for row in result["evidence_rows"]: 
    pprint(row)

답변:
말티즈가 항문을 계속 핥는 행동은 항문낭염이나 염증의 가능성이 있습니다. 이 경우, 항문 부위에 불편함이나 가려움이 있을 수 있습니다. 증상이 지속된다면 빠른 진단과 치료를 위해 동물병원에 상담받는 것이 좋습니다.

근거 데이터 row:
{'meta.department': '내과',
 'meta.disease': '기타',
 'meta.lifeCycle': '성견',
 'qa.output': '말씀해주신 증상을 바탕으로 판단해볼 때, 항문낭염의 가능성이 높아 보입니다. 만약 항문낭염으로 인해 항문낭이 '
              '파열된 상황이 라면, 외와 적인 교정이 필요할 수 있습니다. 병변 부위를 세척하고 소독하는 등 내와 적인 처치를 '
              '통해 염증을 치료할 수 있으나, 재발 가능성이 있기 때문에 각별한 주의가 요구됩니다. 이와 같은 상황에서는 빠른 '
              '진단과 치료가 필요하므로 병원의 상담을 받아보시기를 권장합니다. 기본 상태를 보며 관리는 서서히 유지해 주세요 '
              '자극이 있으면 강도를 낮춰 주세요. 자극을 줄이면서 외용제는 겹겹이 바르지 않도록 주의해 주시고, 필요하면 간격을 '
              '조정해 주시고, 필요하면 간격을 조정해 주세요, 처방 용량과 간격을 지켜 주세요. 필요시 꾸준히 경과를 보며 '
              '강도나 빈도를 조정지켜 주세요 필요하면 간격을 조정해 주세요. 경과 꾸준히 사진을 규칙적으로 남겨 주세요 증상이 '
              '지속되면 내원유지하시고 주 1~2회 정도 확인해 주시고, 필요하면 간격을 조정해 주세요.'}
{'meta.department': '피부과',
 'meta.disease': '외이염',
 'meta.lifeCycle': '성견',
 'qa.output': '강아지가 귀의 가려움증을 경험하고 있다면, 외이염을 겪고 있을 가능성이 상당히 높습니다. 외이염의 원인은 매우 '
           